# 03 · Fine-tuning Experiments (run on Google Colab GPU)

**Runtime → Change runtime type → T4 GPU.** This notebook trains and evaluates every experiment in `configs/experiments.yaml`. Results and model weights are written to **Google Drive**, so a disconnected session resumes where it stopped: experiments that already have `metrics.json` are skipped.

| Priority | Experiments | Research question |
|---|---|---|
| 1 | `zs_qwen05`, `lora_qwen05_r16`, `ft_flant5_base`, `lora_qwen05_r16_rag` | Does domain fine-tuning help? Decoder+LoRA vs encoder-decoder full fine-tuning? Does retrieval help? |
| 2 | `lora_qwen05_r8`, `lora_qwen05_r32`, `lora_qwen05_r16_lr5e5`, `lora_qwen05_r16_attn`, `lora_qwen05_r16_clean`, `zs_qwen05_rag` | Hyperparameter and data ablations: rank, learning rate, target modules, quality filtering |
| 3 | `zs_qwen15`, `lora_qwen15_r16`, `lora_qwen15_r16_rag` | Does a 3× larger base model help? |

Approximate T4 time: about 15–20 min per 0.5B experiment and about 45 min for the 1.5B LoRA run. Prerequisites: add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) and accept the AfriMed-QA terms on Hugging Face.

In [ ]:
!nvidia-smi

In [ ]:
GITHUB_REPO = "https://github.com/mukunzindahiro-max/afrimed-assistant.git"
!git clone -q $GITHUB_REPO /content/afrimed-assistant || (cd /content/afrimed-assistant && git pull -q)
%cd /content/afrimed-assistant
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/afrimed-assistant"
os.makedirs(f"{DRIVE}/results", exist_ok=True)
os.makedirs(f"{DRIVE}/models", exist_ok=True)
!rm -rf results models && ln -s $DRIVE/models models && mkdir -p $DRIVE/results && cp -rn results/* $DRIVE/results/ 2>/dev/null; rm -rf results && ln -s $DRIVE/results results
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path("/content/afrimed-assistant")
sys.path.insert(0, str(ROOT / "src"))
import os
os.environ.setdefault("AFRIMED_ROOT", str(ROOT))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [ ]:
from afrimed.data import prepare
_, sup, unlabeled, report = prepare()
print(json.dumps(report["split_counts"], indent=1))

## Model choices

* **Qwen2.5-0.5B-Instruct / 1.5B-Instruct** (Qwen Team, 2024): decoder-only Transformers with grouped-query attention, RoPE positions and SwiGLU MLPs. They are instruction-tuned, small enough for a free T4 and for CPU deployment, and use a chat template that we keep at fine-tuning time.
* **LoRA** (Hu et al., 2021): freezes the pretrained weights $W$ and learns a low-rank update $\Delta W = \frac{\alpha}{r} BA$ with $B\in\mathbb{R}^{d\times r}$ and $A\in\mathbb{R}^{r\times k}$. With $r=16$ on all linear layers, only about 1.8% of parameters are trained.
* **Flan-T5-base** (Chung et al., 2022): an encoder-decoder Transformer with full fine-tuning, as a contrasting architecture (the sequence-to-sequence model from Week 4).
* **RAG variant:** top-3 nearest training QA pairs, found with MiniLM embeddings, are inserted into the prompt as "Reference notes" (Lewis et al., 2020).

Training objective: causal LM cross-entropy **on the response tokens only**. Evaluation:
* **MCQ:** compare the next-token probabilities of `A`–`E` after the forced prefix `Answer:` (as in MMLU-style evaluation). This avoids parsing failures and gives calibrated probabilities.
* **Multi-answer MCQ:** greedy generation, parse the letters, exact match and Jaccard.
* **SAQ:** greedy generation (≤128 new tokens), scored with ROUGE-L, BLEU, chrF, BERTScore-F1 (rescaled) and MiniLM semantic similarity.

In [ ]:
from afrimed.train import load_experiments
exps = load_experiments()
pd.DataFrame([{k: getattr(c, k) for k in ["name", "priority", "model_id", "method", "lora_r", "lora_alpha", "learning_rate", "epochs", "target_modules", "data_variant", "rag"]} for c in exps.values()])

## Priority 1: core experiments

In [ ]:
from afrimed.experiments import run_all
run_all(max_priority=1)

In [ ]:
from afrimed.plots import load_all_metrics, results_table, plot_loss_curves
display(results_table(load_all_metrics()).round(4))
plot_loss_curves(["lora_qwen05_r16", "ft_flant5_base"]); plt.show()

## Priority 2: ablations (rank, learning rate, target modules, data quality, RAG)

In [ ]:
run_all(max_priority=2)

In [ ]:
display(results_table(load_all_metrics()).round(4))
plot_loss_curves(["lora_qwen05_r8", "lora_qwen05_r16", "lora_qwen05_r32", "lora_qwen05_r16_lr5e5", "lora_qwen05_r16_attn", "lora_qwen05_r16_clean"]); plt.show()

## Priority 3: larger base model (Qwen2.5-1.5B)

In [ ]:
run_all(max_priority=3)

In [ ]:
results_table(load_all_metrics()).round(4)

## Publish the best model to the Hugging Face Hub

The deployed web app downloads the merged model, the retrieval index and the domain gate from the Hub. Set `BEST` to the experiment chosen in the results table (the 0.5B model is recommended for a responsive CPU Space unless the 1.5B model is clearly better).

In [ ]:
BEST = "lora_qwen05_r16"
HUB_REPO = "mukunzindahiro-max/afrimed-assistant-qwen2.5-0.5b-lora"
!python scripts/push_to_hub.py --experiment $BEST --repo $HUB_REPO

## Download results to commit to GitHub

In [ ]:
!cd /content && zip -qr afrimed_results.zip afrimed-assistant/results afrimed-assistant/reports/figures -x '*.ipynb_checkpoints*'
from google.colab import files
files.download("/content/afrimed_results.zip")